# yolov5_cpp — real-GPU check (Colab)

Builds the from-scratch pure engine with `nvcc -DUSE_CUDA` and runs it on an **actual GPU**.
The same source runs on CPU with `g++`. Verifies the `bk::` device seam, the full forward
against PyTorch, and a training loop (forward+backward+update) on the GPU.

**First: Runtime → Change runtime type → GPU (T4 is fine).**


### 1. GPU + nvcc


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2


### 2. Clone


In [ ]:
%cd /content
!rm -rf yolov5_cpp
!git clone -q https://github.com/yomei-o/yolov5_cpp.git
%cd /content/yolov5_cpp


### 3. Seam on the GPU (and CPU, same source)


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m17_gpu.cpp -o m17_gpu && ./m17_gpu
!g++ -std=c++20 -O2 -fopenmp pure/m17_gpu.cpp -o m17_cpu && ./m17_cpu


### 4. Weights + reference (Ultralytics)


In [ ]:
# install deps + generate weights/reference into pure/ref/data_net (show all output)
!pip -q install ultralytics==8.4.104 seaborn pandas
!python pure/ref/export_yolov5.py 64
!echo '--- data_net ---'; ls -la pure/ref/data_net/ 2>&1 | head


### 5. Full forward on the GPU vs PyTorch


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m1_forward.cpp -o fwd_gpu 2>&1 | tail -20
!./fwd_gpu 2>&1 | tail -5


### 6. Train on the GPU (forward + backward + update, all CUDA)


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 pure/m3_train.cpp -o train_gpu 2>&1 | tail -20
!./train_gpu 2>&1 | tail -12


### Success
- Cell 3: `backend: CUDA (GPU)` + `M17 OK`
- Cell 5: `yolov5 M1: PURE ENGINE == yolov5n forward`
- Cell 6: `iter | total box obj cls lr` with the loss dropping → **training runs on the GPU**.

If any nvcc cell errors, paste the output back to Claude to fix.
